# Diagnostic Corrections — velocity 2x2, det-only fiber, honest small-k geodesic

Three self-contained corrections sharing one bootstrap (GRU + refined RSSM):
1. **Velocity 2x2** {linear, MLP} x {single-frame, 2-frame}, both models, all-t and late-t (t>=15).
   Key Q: does single-frame MLP ~= 2-frame MLP (velocity instantaneously nonlinearly readable ->
   "velocity is temporal" RETIRED), or does the 2-frame window genuinely add?
2. **RSSM det-only fiber-collapse refit**: g(pos,vel)->h_det (256-d deterministic block only),
   vs GRU 0.347 and RSSM full-320 0.605.
3. **Honest small-k geodesic (GRU)**: constant-step, LOCAL_K in {16,32,64}, leave-out-neighborhood
   honest local residual (NOT the projection tautology).

Numbered cells `# [N]`, numbered figures `Fig K`. PNGs -> `/tmp/diagnostic_corrections/`.

In [ ]:
# [1] Shared bootstrap: imports + config + load BOTH models + teacher-force + velocities.
import sys, os
sys.path.insert(0, "../../..")   # repo root -> import pim
sys.path.insert(0, "../..")      # notebooks/ -> helpers

from dataclasses import replace
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display
import h5py

import pim.eval as eval
from pim.extractors import LinearExtractor, MLPExtractor, StateDefinition, identity_mse
from pim.editors import (
    probe_decomposition, inject_state,
    fit_state_subspace, project_to_subspace, offmanifold_residual,
    fit_local_subspace,
)
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader

torch.manual_seed(0); np.random.seed(0)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 512
NUM_WORKERS = 6
N_OBJ       = 2
DATA_DIR    = "../../../datasets/4_fixed_refl_inview"
OUT = "/tmp/diagnostic_corrections"; os.makedirs(OUT, exist_ok=True)

GRU_CKPT  = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
RSSM_CKPT = "../../../runs/rssm/4_dset4_refined_best/best_model.pt"

# ---- Data (shared) ----
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
DT = float(test.config["dataset"]["sim"]["sim"]["dt"]) if "sim" in test.config["dataset"].get("sim",{}) else float(test.config["dataset"]["sim"]["dt"])
print("device:", DEVICE, " dt:", DT)

# ---- GRU ----
gru, gru_info = load_checkpoint(GRU_CKPT, device=DEVICE)
H_GRU = gru.hidden_size
preds_gru, states_gru = eval.teacher_force(gru, test_loader, device=DEVICE)   # (N,39,256)
print(f"GRU  : {gru_info.run_name} (epoch {gru_info.epoch}, val_loss={gru_info.val_loss:.5f})  H={H_GRU}  states={states_gru.shape}")

# ---- RSSM (deterministic prior-mean; posterior-mean states via get_hidden_states) ----
rssm, rssm_info = load_checkpoint(RSSM_CKPT, device=DEVICE)
rssm.sample = False
DET, STO = rssm.cfg.det_size, rssm.cfg.stoch_size
H_RSSM = rssm.hidden_size
preds_rssm, states_rssm = eval.teacher_force(rssm, test_loader, device=DEVICE)  # (N,39,320) posterior-mean
print(f"RSSM : {rssm_info.run_name} (epoch {rssm_info.epoch}, val_loss={rssm_info.val_loss:.5f})  H={H_RSSM} (det={DET}, stoch={STO})  states={states_rssm.shape}")
assert DET == 256 and STO == 64 and H_RSSM == 320, (DET, STO, H_RSSM)


In [ ]:
# [2] Velocities read DIRECTLY from HDF5 `velocities`, aligned like positions[:, :-1].
# states_tf[:, t] aligns with positions[:, t] for t in 0..38, so vel uses [:, :-1, :2, :].
v_test = h5py.File(test.h5_path, "r")["velocities"][:, :, :N_OBJ, :].astype(np.float32)   # (N,40,2,2)
vel_tf  = v_test[:, :-1, :, :]                        # (N,39,2,2) aligned with states_tf
pos_tf  = test.positions[:, :-1, :N_OBJ, :]          # (N,39,2,2)
vis_tf  = test.is_visible[:, :-1, :N_OBJ].all(axis=2)  # (N,39) both objects visible

print("velocity temporal std (constant-vel sim -> ~0):", float(v_test.std(axis=1).mean()))
fd = (test.positions[:5, 1:, :N_OBJ, :] - test.positions[:5, :-1, :N_OBJ, :]) / DT
print("|stored vel - finite diff| max (pos noise inflates):", float(np.abs(fd - v_test[:5, :-1]).max()))
print("mean|v|=", float(np.abs(vel_tf).mean()), " vel range=", float(vel_tf.min()), float(vel_tf.max()))

# Flat target arrays.  pos: [x0,y0,x1,y1]  vel: [vx0,vy0,vx1,vy1]
posflat_tf = pos_tf.reshape(*pos_tf.shape[:2], N_OBJ*2)               # (N,39,4)
velflat_tf = vel_tf.reshape(*vel_tf.shape[:2], N_OBJ*2)               # (N,39,4)
T = states_gru.shape[1]

# late-t mask (t>=15) intersected with visibility.
LATE_T = 15
late_mask = np.zeros_like(vis_tf); late_mask[:, LATE_T:] = True
print("all-t visible frac:", float(vis_tf.mean()), " late-t visible frac:", float((vis_tf & late_mask).mean()))


## Section 1 — Velocity 2x2 {linear, MLP} x {single-frame, 2-frame}, BOTH models  [sub-Q2]

Target = 4-dim velocity (vx,vy)x2 objects. For each model, four probes:

| feature | linear | MLP |
|---|---|---|
| single-frame h_t | (a) | (b) |
| 2-frame [h_{t-1}, h_t] | (c) | (d) |

plus **dh = h_t - h_{t-1}** MLP secondary. Run on all-t AND late-t (t>=15).

**Decision rule:** single-frame MLP ~= 2-frame MLP (ΔR2 <~ 0.03) -> velocity instantaneously
nonlinearly readable -> "velocity is temporal" RETIRED. 2-frame MLP >> single-frame MLP ->
temporal claim survives.

In [ ]:
# [3] Probe-fit helper: fit linear/MLP probe feats->velocity, report overall + per-comp R2 vs mean-baseline.
VCOMP = ["vx0","vy0","vx1","vy1"]

def _r2_stats(pred, y):
    # pred,y: (M,4).  overall R2 across all comps + per-comp R2 vs predict-the-mean baseline.
    mu = y.mean(0, keepdims=True)
    ss_res = ((pred - y)**2).sum(0)
    ss_tot = ((y - mu)**2).sum(0)
    r2_pc = 1 - ss_res / np.clip(ss_tot, 1e-12, None)      # (4,)
    r2_overall = float(1 - ss_res.sum() / ss_tot.sum())
    rmse = float(np.sqrt(((pred - y)**2).mean()))
    return r2_overall, r2_pc, rmse

def fit_vel_probe(feats_tf, y_tf, mask, kind, mlp_hidden=256, n_epochs=100, lr=2e-3, seed=0):
    """feats_tf:(N,Tf,F) y_tf:(N,Tf,4) mask:(N,Tf) bool.  Returns dict of metrics on masked entries."""
    torch.manual_seed(seed); np.random.seed(seed)
    X = feats_tf[mask].astype(np.float32); Y = y_tf[mask].astype(np.float32)
    Din = X.shape[1]
    Xt = torch.from_numpy(X).to(DEVICE); Yt = torch.from_numpy(Y).to(DEVICE)
    if kind == "linear":
        Xa = torch.cat([Xt, torch.ones(Xt.shape[0],1,device=DEVICE)],1)
        sol = torch.linalg.lstsq(Xa, Yt).solution
        with torch.no_grad(): pred = (Xa @ sol).cpu().numpy()
    else:
        net = nn.Sequential(nn.Linear(Din,mlp_hidden), nn.ReLU(),
                            nn.Linear(mlp_hidden,mlp_hidden), nn.ReLU(),
                            nn.Linear(mlp_hidden,4)).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        bs = 4096; Nn = Xt.shape[0]
        for ep in range(n_epochs):
            perm = torch.randperm(Nn, device=DEVICE)
            for i in range(0, Nn, bs):
                idx = perm[i:i+bs]
                loss = ((net(Xt[idx]) - Yt[idx])**2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): pred = net(Xt).cpu().numpy()
    r2o, r2pc, rmse = _r2_stats(pred, Y)
    return dict(r2=r2o, r2pc=r2pc, rmse=rmse, n=X.shape[0])
print("probe helper ready")


In [ ]:
# [4] Build feature sets for a given states_tf.  Returns single-frame, 2-frame window, dh, and aligned y/mask.
def build_feature_sets(states_tf, restrict_late=False):
    # single frame: feats = states_tf[:, 1:], target vel[:, 1:], so it aligns with 2-frame target.
    sf     = states_tf[:, 1:, :]                                   # (N,T-1,H)  h_t (t>=1)
    win    = np.concatenate([states_tf[:, :-1, :], states_tf[:, 1:, :]], -1)  # (N,T-1,2H) [h_{t-1},h_t]
    dh     = states_tf[:, 1:, :] - states_tf[:, :-1, :]            # (N,T-1,H)
    y      = velflat_tf[:, 1:, :]                                  # (N,T-1,4) vel at frame t
    mask   = vis_tf[:, 1:] & vis_tf[:, :-1]                        # both frames visible
    if restrict_late:
        lm = np.zeros_like(mask); lm[:, LATE_T-1:] = True          # t>=15 means index>=15 in orig -> t-1>=14
        mask = mask & lm
    return dict(sf=sf, win=win, dh=dh, y=y, mask=mask)

def run_2x2(states_tf, label, restrict_late=False):
    fs = build_feature_sets(states_tf, restrict_late)
    out = {}
    out[("sf","linear")]  = fit_vel_probe(fs["sf"],  fs["y"], fs["mask"], "linear")
    out[("sf","mlp")]     = fit_vel_probe(fs["sf"],  fs["y"], fs["mask"], "mlp")
    out[("win","linear")] = fit_vel_probe(fs["win"], fs["y"], fs["mask"], "linear")
    out[("win","mlp")]    = fit_vel_probe(fs["win"], fs["y"], fs["mask"], "mlp")
    out[("dh","mlp")]     = fit_vel_probe(fs["dh"],  fs["y"], fs["mask"], "mlp")
    out[("dh","linear")]  = fit_vel_probe(fs["dh"],  fs["y"], fs["mask"], "linear")
    return out

def print_2x2(out, label):
    print(f"\n=== VELOCITY 2x2 — {label}  (n={out[('sf','linear')]['n']}) ===")
    print(f"{'feature':22s} {'linear R2':>10s} {'MLP R2':>10s}")
    for feat, name in [("sf","single-frame h_t"), ("win","2-frame [h-1,h]")]:
        print(f"{name:22s} {out[(feat,'linear')]['r2']:10.4f} {out[(feat,'mlp')]['r2']:10.4f}")
    print(f"{'dh=h_t-h_{t-1}':22s} {out[('dh','linear')]['r2']:10.4f} {out[('dh','mlp')]['r2']:10.4f}")
    d = out[("win","mlp")]["r2"] - out[("sf","mlp")]["r2"]
    verdict = "RETIRE temporal (single-frame MLP ~= 2-frame MLP)" if d <= 0.03 else "temporal SURVIVES (2-frame MLP adds)"
    print(f"  Δ(2frame MLP - single MLP) = {d:+.4f}   ->  {verdict}")
    return d
print("2x2 runners ready")


In [ ]:
# [5] Run the full 2x2 for BOTH models, all-t AND late-t.
res_vel = {}
for mdl_lbl, st in [("GRU", states_gru), ("RSSM(det+stoch)", states_rssm)]:
    res_vel[(mdl_lbl,"all")]  = run_2x2(st, mdl_lbl, restrict_late=False)
    res_vel[(mdl_lbl,"late")] = run_2x2(st, mdl_lbl, restrict_late=True)

deltas = {}
for mdl_lbl in ["GRU","RSSM(det+stoch)"]:
    for reg in ["all","late"]:
        deltas[(mdl_lbl,reg)] = print_2x2(res_vel[(mdl_lbl,reg)], f"{mdl_lbl} / {reg}-t")


In [ ]:
# [6] Per-component R2 table (late-t) for both models, single vs 2-frame MLP.
print("=== PER-COMPONENT velocity R2 (late-t, MLP) ===")
print(f"{'model':18s} {'feat':16s} " + " ".join(f"{c:>7s}" for c in VCOMP))
for mdl_lbl in ["GRU","RSSM(det+stoch)"]:
    for feat, nm in [("sf","single h_t"),("win","2-frame")]:
        pc = res_vel[(mdl_lbl,"late")][(feat,"mlp")]["r2pc"]
        print(f"{mdl_lbl:18s} {nm:16s} " + " ".join(f"{v:7.3f}" for v in pc))


In [ ]:
# [7] Fig 1 — Velocity 2x2 overall R2 by feature set, GRU vs RSSM, all-t & late-t.
plt.style.use("default")
OK = {"blue":"#0072B2","orange":"#D55E00","green":"#009E73","grey":"#999999"}
def style_ax(ax):
    ax.spines[["top","right"]].set_visible(False); ax.grid(alpha=0.25, axis="y")
    ax.axhline(0,color="k",lw=0.8); ax.axhline(1,color="0.6",ls=":",lw=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
featsets = [("sf","linear"),("sf","mlp"),("win","linear"),("win","mlp"),("dh","mlp")]
xl = ["lin\nh_t","MLP\nh_t","lin\n[h-1,h]","MLP\n[h-1,h]","MLP\ndh"]
for j,(mdl_lbl,ax) in enumerate(zip(["GRU","RSSM(det+stoch)"], axes)):
    x = np.arange(len(featsets)); w = 0.38
    va = [res_vel[(mdl_lbl,"all")][k]["r2"] for k in featsets]
    vl = [res_vel[(mdl_lbl,"late")][k]["r2"] for k in featsets]
    ax.bar(x-w/2, va, w, label="all-t", color=OK["blue"])
    ax.bar(x+w/2, vl, w, label="late-t (t>=15)", color=OK["orange"])
    ax.set_xticks(x); ax.set_xticklabels(xl, fontsize=8)
    ax.set_ylabel("overall velocity R2"); style_ax(ax)
    ax.set_title(f"({'a' if j==0 else 'b'}) {mdl_lbl}: velocity recoverability")
    ax.legend(fontsize=8, loc="lower right")
    for xi,vv in zip(x-w/2,va): ax.text(xi, vv+0.01, f"{vv:.2f}", ha="center", fontsize=7)
    for xi,vv in zip(x+w/2,vl): ax.text(xi, vv+0.01, f"{vv:.2f}", ha="center", fontsize=7)
fig.suptitle("Fig 1 — Velocity 2x2: linear vs MLP, single-frame vs 2-frame window (both models)", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/1_velocity_2x2.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved 1_velocity_2x2.png")


## Section 2 — Det-only fiber-collapse refit, RSSM  [sub-Q2]

Refit g(pos,vel) -> h_det (deterministic **256-d block only**), linear + MLP. Report residual
fraction ‖h_det - g‖/‖h_det‖ and R2(h_det). Compare head-to-head: GRU h (0.347), RSSM full-320
(0.605), RSSM det-only (new). Also fit g(pos,vel)->s (stochastic 64) to show how much of 0.605 was s.

**Decision:** is the RSSM's deterministic block as canonical as the GRU's h, or still less canonical?

In [ ]:
# [8] g:(pos,vel)->target fit (linear + MLP), residual fraction + R2.  Generic over target block.
def fit_g(inp_tf, tgt_tf, mask, kind, hidden=512, n_epochs=120, lr=1.5e-3, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    X = inp_tf[mask].astype(np.float32); Y = tgt_tf[mask].astype(np.float32)
    Din, Dout = X.shape[1], Y.shape[1]
    Xt = torch.from_numpy(X).to(DEVICE); Yt = torch.from_numpy(Y).to(DEVICE)
    if kind == "linear":
        Xa = torch.cat([Xt, torch.ones(Xt.shape[0],1,device=DEVICE)],1)
        sol = torch.linalg.lstsq(Xa, Yt).solution
        with torch.no_grad(): pred = Xa @ sol
    else:
        net = nn.Sequential(nn.Linear(Din,hidden), nn.ReLU(),
                            nn.Linear(hidden,hidden), nn.ReLU(),
                            nn.Linear(hidden,Dout)).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        bs = 4096; Nn = Xt.shape[0]
        for ep in range(n_epochs):
            perm = torch.randperm(Nn, device=DEVICE)
            for i in range(0, Nn, bs):
                idx = perm[i:i+bs]
                loss = ((net(Xt[idx]) - Yt[idx])**2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): pred = net(Xt)
    resid2 = ((pred - Yt)**2).sum()
    tgt_norm2 = (Yt**2).sum()
    res_frac = float((resid2 / tgt_norm2).sqrt())               # ||h-g|| / ||h||
    mu = Yt.mean(0, keepdim=True)
    r2 = float(1 - resid2 / ((Yt-mu)**2).sum())
    return res_frac, r2

# Input = 8-d (pos,vel). Targets: RSSM h_det (256), RSSM s_stoch (64), RSSM full (320) [reproduce 0.605].
posvel_tf = np.concatenate([posflat_tf, velflat_tf], -1)          # (N,39,8)
h_det   = states_rssm[..., :DET]                                  # (N,39,256)
s_stoch = states_rssm[..., DET:]                                  # (N,39,64)
h_full  = states_rssm                                            # (N,39,320)

blocks = {"RSSM h_det (256)": h_det, "RSSM s_stoch (64)": s_stoch,
          "RSSM full (320)": h_full, "GRU h (256)": states_gru}
res_fiber = {}
print(f"{'target block':22s} {'lin resid':>10s} {'lin R2':>9s} {'MLP resid':>10s} {'MLP R2':>9s}")
for name, blk in blocks.items():
    mask = vis_tf
    lrf, lr2 = fit_g(posvel_tf, blk, mask, "linear")
    mrf, mr2 = fit_g(posvel_tf, blk, mask, "mlp")
    res_fiber[name] = dict(lin=(lrf,lr2), mlp=(mrf,mr2))
    print(f"{name:22s} {lrf:10.4f} {lr2:9.4f} {mrf:10.4f} {mr2:9.4f}")


In [ ]:
# [9] Section 2 head-to-head verdict table.  Fiber = MLP residual fraction (nonlinear g).
print("=== FIBER-COLLAPSE HEAD-TO-HEAD (MLP g, residual fraction ||blk-g||/||blk||) ===")
print(f"{'block':22s} {'MLP resid frac':>15s}  (lower = more canonical / (pos,vel) determines it)")
for name in ["GRU h (256)","RSSM full (320)","RSSM h_det (256)","RSSM s_stoch (64)"]:
    print(f"{name:22s} {res_fiber[name]['mlp'][0]:15.4f}")
gru_r  = res_fiber["GRU h (256)"]["mlp"][0]
det_r  = res_fiber["RSSM h_det (256)"]["mlp"][0]
full_r = res_fiber["RSSM full (320)"]["mlp"][0]
s_r    = res_fiber["RSSM s_stoch (64)"]["mlp"][0]
print(f"\nGRU h = {gru_r:.4f}   RSSM full-320 = {full_r:.4f}   RSSM det-only = {det_r:.4f}   (s-block = {s_r:.4f})")
print(f"stochastic s inflates full residual by: full {full_r:.3f} vs det {det_r:.3f} (Δ={full_r-det_r:+.3f})")
if det_r <= gru_r + 0.03:
    print(f"VERDICT: RSSM det block AS canonical (or more) than GRU h  (det {det_r:.3f} <~ GRU {gru_r:.3f})")
elif det_r < full_r:
    print(f"VERDICT: RSSM det block MORE canonical than full-320 but LESS than GRU  (det {det_r:.3f} > GRU {gru_r:.3f})")
else:
    print(f"VERDICT: det block still less canonical than GRU  (det {det_r:.3f} vs GRU {gru_r:.3f})")


In [ ]:
# [10] Fig 2 — Fiber-collapse residual fraction (a) & R2 (b): GRU vs RSSM full vs det vs s.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
order = ["GRU h (256)","RSSM full (320)","RSSM h_det (256)","RSSM s_stoch (64)"]
cols  = [OK["green"], OK["grey"], OK["blue"], OK["orange"]]
x = np.arange(len(order)); w = 0.38
ax = axes[0]
lin_rf = [res_fiber[n]["lin"][0] for n in order]; mlp_rf = [res_fiber[n]["mlp"][0] for n in order]
ax.bar(x-w/2, lin_rf, w, label="linear g", color=OK["blue"])
ax.bar(x+w/2, mlp_rf, w, label="MLP g",    color=OK["orange"])
for xi,v in zip(x+w/2, mlp_rf): ax.text(xi, v+0.01, f"{v:.3f}", ha="center", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("residual fraction ||blk-g||/||blk||"); style_ax(ax)
ax.set_title("(a) fiber collapse: is block a function of (pos,vel)?"); ax.legend(fontsize=8)
ax = axes[1]
lin_r2 = [res_fiber[n]["lin"][1] for n in order]; mlp_r2 = [res_fiber[n]["mlp"][1] for n in order]
ax.bar(x-w/2, lin_r2, w, label="linear g", color=OK["blue"])
ax.bar(x+w/2, mlp_r2, w, label="MLP g",    color=OK["orange"])
ax.set_xticks(x); ax.set_xticklabels(order, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("R2 on block"); style_ax(ax)
ax.set_title("(b) R2 explained by g(pos,vel)"); ax.legend(fontsize=8)
fig.suptitle("Fig 2 — Det-only fiber refit: RSSM deterministic block vs GRU h vs full-320 vs stochastic s", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/2_fiber_detonly.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved 2_fiber_detonly.png")


## Section 3 — Honest small-k geodesic, GRU  [sub-Q1/3]

Constant-step geodesic walk toward the position-probe target, LOCAL_K in {16,32,64} (vs old 512).
At each k:
- readout RMSE reached?
- **honest local off-manifold residual** via leave-out neighborhood (fit tangent on k neighbors
  EXCLUDING the query point) — NOT the projection tautology. Expect ~0.75-0.84, not ~0.
- does the observation move (obs change vs unsteered, % of full-state-swap)?

1D-line waterfalls (green=target, red=ghost) for ~3 samples.

In [ ]:
# [11] Geodesic setup on the GRU: position probe, warm-up, target, honest leave-out residual.
model = gru; H = H_GRU
states_tf = states_gru                                # module-level name reused by helpers
SUBSPACE_VAR, LOCAL_VAR, LOCAL_BANK_SIZE = 0.90, 0.90, 50_000
N_GEO, K_ITERS = 400, 120

# position probe: h -> (N_OBJ,2) positions (fit on teacher-forced states).
sdef = StateDefinition(name="pos", state_shape=(N_OBJ,2), extract_fn=lambda b: b["x"][:, :N_OBJ, :])
linear_pos = LinearExtractor(H, sdef, use_lstsq=True)
linear_pos.fit(states_tf, test.positions[:, :-1, :N_OBJ, :], mask=vis_tf, device=DEVICE)
linear_pos = linear_pos.to(DEVICE).eval()

# global subspace + bank
subspace = fit_state_subspace(states_tf, var_threshold=SUBSPACE_VAR)
subspace_dev = replace(subspace, mean=subspace.mean.to(DEVICE), basis=subspace.basis.to(DEVICE),
                       explained_variance_ratio=subspace.explained_variance_ratio.to(DEVICE))
_bank_all = states_tf.reshape(-1, H)
_sub = np.random.RandomState(0).choice(_bank_all.shape[0], size=min(LOCAL_BANK_SIZE,_bank_all.shape[0]), replace=False)
bank_dev = torch.from_numpy(_bank_all[_sub]).float().to(DEVICE)

# ---- HONEST leave-out-neighborhood local residual ----
# For a query h, find k+1 nearest bank pts, DROP the nearest (self if in bank), fit PCA on the rest,
# then measure ||h - proj|| and the FRACTION ||h-proj|| / ||h - local_mean||.  NOT the projection tautology.
from pim.editors.manifold_steering import _pca_subspace
@torch.no_grad()
def honest_local_resid(h_batch, k_neighbors, leave_out=True, n_probe=100, var_threshold=LOCAL_VAR):
    hb = h_batch if isinstance(h_batch, torch.Tensor) else torch.as_tensor(h_batch, device=DEVICE, dtype=torch.float32)
    raws, fracs = [], []
    npb = min(n_probe, hb.shape[0])
    for i in range(npb):
        q = hb[i].reshape(-1)
        d = torch.cdist(q[None], bank_dev)[0]
        kk = k_neighbors + (1 if leave_out else 0)
        idx = torch.topk(d, min(kk, bank_dev.shape[0]), largest=False).indices
        if leave_out: idx = idx[1:]                              # drop nearest (self)
        nbrs = bank_dev[idx]
        sub = _pca_subspace(nbrs, n_components=None, var_threshold=var_threshold)
        proj = project_to_subspace(q[None], sub)[0]
        raw = float((q - proj).norm())
        denom = float((q - sub.mean).norm())
        raws.append(raw); fracs.append(raw / max(denom, 1e-9))
    return float(np.mean(raws)), float(np.mean(fracs))

# real-state honest residual reference at each k (query points ARE real states -> leave_out matters most here).
real_states_probe = torch.from_numpy(_bank_all[_sub[:200]]).float().to(DEVICE)
print("real-state HONEST local residual (raw, frac) by k:")
real_ref = {}
for k in [16,32,64]:
    real_ref[k] = honest_local_resid(real_states_probe, k, leave_out=True, n_probe=200)
    # tautology version (self INCLUDED) for contrast:
    taut = honest_local_resid(real_states_probe, k, leave_out=False, n_probe=200)
    print(f"  k={k:3d}: honest raw={real_ref[k][0]:.4f} frac={real_ref[k][1]:.4f}   | tautology(self-in) raw={taut[0]:.4f} frac={taut[1]:.4f}")


In [ ]:
# [12] Warm-up to edit frame, target readout, probe decomposition, constant-step geodesic walk fn.
from tqdm.auto import tqdm
N = min(N_GEO, edits.n_samples)
warm = eval.warm_up_to_edit(model, edits.obs[:N], edits.edit_frame, n_viz=N, n_ctx_show=8, device=DEVICE)
h_base = warm.h_at_edit[:N]
targets = edits.positions[:N, edits.edit_frame, :N_OBJ, :].reshape(N, N_OBJ*2)

A, b, A_pinv = probe_decomposition(linear_pos)
h0  = torch.from_numpy(h_base).float().to(DEVICE)
tgt = torch.from_numpy(targets).float().to(DEVICE)
def readout(h):      return h @ A.T + b
def readout_rmse(h): return float((readout(h) - tgt).pow(2).mean().sqrt())

@torch.no_grad()
def geodesic_const(h_start, target, k_local, const_step, k_iters=K_ITERS, var_threshold=LOCAL_VAR):
    Nn = h_start.shape[0]
    h_out = torch.empty_like(h_start)
    rmse_log = np.zeros((Nn, k_iters+1)); step_log = np.zeros((Nn, k_iters))
    for i in tqdm(range(Nn), desc=f"geo k={k_local}", leave=False):
        h = h_start[i:i+1]; t = target[i:i+1]
        rmse_log[i,0] = float((readout(h)-t).pow(2).mean().sqrt())
        for kk in range(k_iters):
            h_full = inject_state(h, t, A, A_pinv, b)
            d = h_full - h; nrm = d.norm()
            dhat = d/nrm if float(nrm)>1e-12 else d
            h_step = h + const_step * dhat
            sub = fit_local_subspace(bank_dev, h_step[0], k_neighbors=k_local,
                                     var_threshold=var_threshold, bank_size=LOCAL_BANK_SIZE)
            h_proj = project_to_subspace(h_step, sub)
            step_log[i,kk] = float((h_proj - h).norm()); h = h_proj
            rmse_log[i,kk+1] = float((readout(h)-t).pow(2).mean().sqrt())
        h_out[i] = h[0]
    return h_out, {"rmse":rmse_log, "step":step_log}

print(f"N={N}  edit_frame={edits.edit_frame}  cold-start readout RMSE={readout_rmse(h0):.4f}")


In [ ]:
# [13] Run constant-step geodesic at LOCAL_K in {16,32,64}. Const step matches the k150 reference:
# STEP_FRAC * (mean one-shot pseudoinverse jump norm) — i.e. the first-fractional-iteration step size.
# (An earlier draft used a ~10x smaller step; the walk then barely moved and its low residual was an
#  artifact of NOT traversing, not evidence about the manifold. Fixed to the reference step size.)
STEP_FRAC = 0.34
with torch.no_grad():
    d0 = (inject_state(h0, tgt, A, A_pinv, b) - h0).norm(dim=-1)
CONST_STEP = STEP_FRAC * float(d0.mean())         # ~0.18: same per-step magnitude as geodesic_walk_k150
print(f"mean full pseudoinv jump norm = {float(d0.mean()):.4f}  (STEP_FRAC={STEP_FRAC}) -> CONST_STEP = {CONST_STEP:.4f}")
print(f"total available path length over K={K_ITERS} iters = {CONST_STEP*K_ITERS:.2f} (jump norm ~{float(d0.mean()):.2f})")

geo_res = {}
for k in [16,32,64]:
    hk, logk = geodesic_const(h0, tgt, k, CONST_STEP)
    hraw, hfrac = honest_local_resid(hk, k, leave_out=True, n_probe=min(150,N))
    geo_res[k] = dict(h=hk, log=logk, honest_raw=hraw, honest_frac=hfrac,
                      final_rmse=float(logk["rmse"][:,-1].mean()),
                      cold_rmse=float(logk["rmse"][:,0].mean()))
    print(f"k={k:3d}: final readout RMSE={geo_res[k]['final_rmse']:.4f} (cold={geo_res[k]['cold_rmse']:.4f})  "
          f"honest local resid raw={hraw:.4f} frac={hfrac:.4f}  (real-frac ref={real_ref[k][1]:.4f})")


In [ ]:
# [14] Rollout each k's edited state + unsteered + full-state-swap; measure obs change.
N_ROLLOUT = 8
@torch.no_grad()
def rollout_from_flat(h_array, n_rollout):
    obs_all=[]
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o,_ = _rollout(model, h, n_rollout); obs_all.append(o)
    return np.stack(obs_all)

# Full-state-swap reference: teacher-force the edits obs up to the POST-edit target to get a "true" swapped h.
# Cheaper proxy for the 100% scale: the pseudoinverse one-shot injection (fully reaches readout, off-manifold).
h_swap = inject_state(h0, tgt, A, A_pinv, b)

variant_h = {"unsteered": h0, "full-swap (pinv)": h_swap}
for k in [16,32,64]: variant_h[f"geo k={k}"] = geo_res[k]["h"]
roll_obs = {n: rollout_from_flat(h.detach().cpu().numpy(), N_ROLLOUT) for n,h in variant_h.items()}
OBS_RES = roll_obs["unsteered"].shape[-1]

def rms(a,b): return float(np.sqrt(((a-b)**2).mean()))
obs_u = roll_obs["unsteered"][:, 0, :]
swap_change = rms(roll_obs["full-swap (pinv)"][:,0,:], obs_u)     # 100% reference
print(f"full-state-swap obs change (step0) = {swap_change:.4f}  (= 100% reference)")
print(f"{'variant':18s} {'obs change':>11s} {'% of swap':>10s}")
obs_change_tbl={}
for n in ["geo k=16","geo k=32","geo k=64"]:
    ch = rms(roll_obs[n][:,0,:], obs_u); pct = 100*ch/max(swap_change,1e-9)
    obs_change_tbl[n]=(ch,pct)
    print(f"{n:18s} {ch:11.4f} {pct:10.1f}")


In [ ]:
# [15] Fig 3 — Section 3 summary: (a) readout RMSE convergence per-k, (b) honest resid vs real ref, (c) obs change %.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
ax = axes[0]
kcols = {16:OK["blue"],32:OK["orange"],64:OK["green"]}
for k in [16,32,64]:
    rm = geo_res[k]["log"]["rmse"].mean(0)
    ax.plot(rm, color=kcols[k], lw=2, label=f"k={k} (final {rm[-1]:.3f})")
ax.axhline(geo_res[16]["cold_rmse"], color="k", ls=":", lw=1, label="cold start")
ax.set_xlabel("geodesic iter"); ax.set_ylabel("readout RMSE"); style_ax(ax)
ax.set_title("(a) does readout reach target?"); ax.legend(fontsize=8)
ax = axes[1]
x = np.arange(3); w=0.38
hf = [geo_res[k]["honest_frac"] for k in [16,32,64]]
rf = [real_ref[k][1] for k in [16,32,64]]
ax.bar(x-w/2, hf, w, label="walked (honest)", color=OK["orange"])
ax.bar(x+w/2, rf, w, label="real states (honest)", color=OK["green"])
ax.axhspan(0.75,0.84, color="0.85", alpha=0.5, label="expected 0.75-0.84")
for xi,v in zip(x-w/2,hf): ax.text(xi,v+0.01,f"{v:.2f}",ha="center",fontsize=7)
ax.set_xticks(x); ax.set_xticklabels([f"k={k}" for k in [16,32,64]])
ax.set_ylabel("honest local resid FRACTION"); style_ax(ax)
ax.set_title("(b) honest off-manifold residual (leave-out)"); ax.legend(fontsize=7)
ax = axes[2]
pct = [obs_change_tbl[f"geo k={k}"][1] for k in [16,32,64]]
ax.bar([f"k={k}" for k in [16,32,64]], pct, color=[kcols[k] for k in [16,32,64]])
for xi,v in enumerate(pct): ax.text(xi,v+1,f"{v:.0f}%",ha="center",fontsize=8)
ax.set_ylabel("obs change (% of full-swap)"); style_ax(ax); ax.axhline(100,color="0.6",ls=":")
ax.set_title("(c) does the observation move?")
fig.suptitle("Fig 3 — Honest small-k geodesic (GRU): reachability, honest residual, obs movement", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/3_geodesic_smallk.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved 3_geodesic_smallk.png")


In [ ]:
# [16] Fig 4 — 1D-line waterfalls (3 samples x variants). Green=target obj loc, red=ghost loc.
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
sim = test.config["dataset"]["sim"]
def make_cfg(nf):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                     n_objects=N_OBJ, radius=sim["radius"], n_frames=nf, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                     obs_noise_std=0.0, boundary="open", always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], dtype=np.float32)
RAD  = np.array([sim["radius"]]*N_OBJ, dtype=np.float32)
COLc = np.tile(np.array([[1,1,1]], np.float32), (N_OBJ,1))
tgt_pos = edits.positions[:N, edits.edit_frame, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, edits.edit_frame-1, :N_OBJ, :].astype(np.float32)
cfg1 = make_cfg(1)
tgt_render_id = np.zeros((N,OBS_RES), np.int64); pre_render_id = np.zeros((N,OBS_RES), np.int64)
for i in range(N):
    sc = Scene(positions=tgt_pos[i][None], velocities=np.zeros((1,N_OBJ,2),np.float32),
               radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, rid, _ = render_scene(sc); tgt_render_id[i]=rid[0]
    scp = Scene(positions=pre_pos[i][None], velocities=np.zeros((1,N_OBJ,2),np.float32),
                radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, ridp, _ = render_scene(scp); pre_render_id[i]=ridp[0]
edit_obj = edits.edit_object[:N]

SAMPLES = [0,1,2]
order = ["unsteered","geo k=16","geo k=32","geo k=64","full-swap (pinv)"]
def centroid(m): idx=np.where(m)[0]; return idx.mean() if idx.size else np.nan
fig, axes = plt.subplots(len(SAMPLES), len(order), figsize=(2.5*len(order), 3.0*len(SAMPLES)), squeeze=False)
for r,smp in enumerate(SAMPLES):
    tgt_cx = centroid(tgt_render_id[smp]==edit_obj[smp]); pre_cx = centroid(pre_render_id[smp]==edit_obj[smp])
    for c,n in enumerate(order):
        ax = axes[r][c]
        ax.imshow(roll_obs[n][smp], aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", lw=1.4, alpha=0.9)
        if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.4, alpha=0.9)
        if r==0: ax.set_title(n, fontsize=9)
        if c==0: ax.set_ylabel(f"smp {smp}\nframe", fontsize=9)
        ax.set_xlabel("ray", fontsize=8)
axes[0][0].plot([],[],color="#00E676",lw=2,label="target loc"); axes[0][0].plot([],[],color="#FF5252",ls="--",lw=2,label="ghost loc")
axes[0][0].legend(loc="upper right", fontsize=6)
fig.suptitle("Fig 4 — Honest small-k geodesic waterfalls: green=where edited obj SHOULD be, red=ghost\n"
             "(good edit: bright streak at green, nothing at red)", y=1.015, fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUT}/4_waterfalls.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved 4_waterfalls.png")


In [ ]:
# [17] AUTO VERDICT — all three sections.
print("="*78); print("DIAGNOSTIC CORRECTIONS — AUTO VERDICT"); print("="*78)

print("\n[SECTION 1] VELOCITY 2x2 — does the temporal window add over single-frame MLP?")
for mdl_lbl in ["GRU","RSSM(det+stoch)"]:
    for reg in ["all","late"]:
        o = res_vel[(mdl_lbl,reg)]; d = deltas[(mdl_lbl,reg)]
        v = "RETIRE temporal" if d<=0.03 else "temporal survives"
        print(f"  {mdl_lbl:16s}/{reg:4s}: sf-MLP={o[('sf','mlp')]['r2']:.3f}  2f-MLP={o[('win','mlp')]['r2']:.3f}  "
              f"Δ={d:+.3f}  sf-lin={o[('sf','linear')]['r2']:.3f} -> {v}")
gd = deltas[("GRU","late")]
print(f"  >> GRU late-t key: single-frame MLP vs 2-frame MLP Δ={gd:+.3f} "
      f"({'velocity is instantaneously nonlinearly readable -> RETIRE' if gd<=0.03 else 'temporal genuinely adds'})")

print("\n[SECTION 2] DET-ONLY FIBER (MLP residual fraction, lower=more canonical):")
print(f"  GRU h={gru_r:.3f}   RSSM full-320={full_r:.3f}   RSSM det-only={det_r:.3f}   (s-block={s_r:.3f})")
print(f"  det-only vs GRU: {'det <= GRU (as/more canonical)' if det_r<=gru_r+0.03 else 'det > GRU (less canonical/more curved)'}")

print("\n[SECTION 3] HONEST SMALL-K GEODESIC (GRU):")
for k in [16,32,64]:
    g = geo_res[k]
    reached = "REACHES" if g["final_rmse"]<0.5*g["cold_rmse"] else "partial/plateau"
    obsp = obs_change_tbl[f"geo k={k}"][1]
    print(f"  k={k:2d}: readout {g['cold_rmse']:.3f}->{g['final_rmse']:.3f} ({reached})  "
          f"honest resid frac={g['honest_frac']:.3f} (real {real_ref[k][1]:.3f})  obs moves {obsp:.0f}% of swap")
print(f"\nPNGs: {OUT}/{{1_velocity_2x2,2_fiber_detonly,3_geodesic_smallk,4_waterfalls}}.png")
